# Customer Segmentation on Credit Card Behaviour
### Unsupervised Learning: Preprocessing → PCA → Clustering → Profiling

**Dataset:** `CC GENERAL.csv` — 8,950 active credit-card holders, 17 behavioural variables observed over ~12 months.

**Goal:** there is no target column here. We want to discover natural groups of customers from their spending, cash-advance, and repayment behaviour, so the segments can drive marketing strategy.

**Pipeline**
1. Load data & exploratory analysis
2. Preprocessing — missing values, feature engineering, skew correction, scaling
3. PCA — dimensionality reduction & visualisation
4. K-Means — choosing *k* with elbow / silhouette / Davies-Bouldin / Calinski-Harabasz
5. Hierarchical clustering + dendrogram
6. DBSCAN (density based)
7. Gaussian Mixture Model
8. Model comparison
9. Cluster profiling & business personas
10. Export model and labelled data

## 0. Setup

In [ ]:
# Colab already has all of these. Uncomment if a package is missing.
# !pip install -q scikit-learn pandas numpy matplotlib seaborn scipy joblib

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, calinski_harabasz_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 12

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

print('Setup complete.')

## 1. Load the data

In [ ]:
CSV_NAME = 'CC GENERAL.csv'

def load_data(name=CSV_NAME):
    # 1) file sitting next to the notebook
    import os
    if os.path.exists(name):
        return pd.read_csv(name)
    # 2) Google Drive mount
    drive_path = f'/content/drive/MyDrive/{name}'
    if os.path.exists(drive_path):
        return pd.read_csv(drive_path)
    # 3) manual upload in Colab
    try:
        from google.colab import files
        print('Upload the CSV file...')
        uploaded = files.upload()
        return pd.read_csv(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(f'Could not find {name}. Put it beside the notebook.')

df_raw = load_data()
df = df_raw.copy()
print('Shape:', df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
# Statistical summary. Note the huge gap between the 75th percentile and max on most
# columns -> heavy right skew, which we deal with in preprocessing.
df.describe().T

In [ ]:
# --- Missing values ---
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_tbl = pd.DataFrame({'missing': missing, 'percent': missing_pct})
print(miss_tbl)

# --- Duplicates ---
print('\nDuplicate rows      :', df.duplicated().sum())
print('Duplicate CUST_ID   :', df['CUST_ID'].duplicated().sum())

if len(miss_tbl):
    ax = miss_tbl['percent'].plot(kind='barh', color='#d1495b')
    ax.set_title('Missing values (% of rows)')
    ax.set_xlabel('%')
    plt.tight_layout(); plt.show()

In [ ]:
# Distribution of every numeric feature
num_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(6, 3, figsize=(16, 20))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col].dropna(), bins=50, kde=True, ax=ax, color='#3a7ca5')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
for ax in axes.ravel()[len(num_cols):]:
    ax.set_visible(False)
plt.suptitle('Raw feature distributions', fontsize=15, fontweight='bold', y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
# Skewness: anything above ~1 is strongly right-skewed
skew = df[num_cols].skew().sort_values(ascending=False)
plt.figure(figsize=(9, 6))
sns.barplot(x=skew.values, y=skew.index, hue=skew.index, legend=False, palette='coolwarm')
plt.axvline(1, ls='--', c='k', lw=1)
plt.axvline(-1, ls='--', c='k', lw=1)
plt.title('Skewness by feature (dashed lines = |skew| of 1)')
plt.tight_layout(); plt.show()
skew.to_frame('skew')

In [ ]:
# Correlation structure - tells us how much redundancy PCA will be able to squeeze out
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(13, 10))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 7}, square=True, linewidths=.5, cbar_kws={'shrink': .7})
plt.title('Correlation matrix')
plt.tight_layout(); plt.show()

# strongest pairs
pairs = corr.where(~mask).stack().sort_values(key=abs, ascending=False)
print('Most correlated pairs:')
print(pairs.head(10))

**Read-out from the EDA**

* `MINIMUM_PAYMENTS` (313 rows) and `CREDIT_LIMIT` (1 row) have missing values.
* Almost every monetary column is extremely right-skewed — a small number of very heavy spenders dominate. Distance-based algorithms like K-Means would be driven almost entirely by these outliers, so we log-transform.
* Several features are strongly correlated (e.g. `PURCHASES` with `ONEOFF_PURCHASES`, `CASH_ADVANCE` with `CASH_ADVANCE_TRX`). That redundancy is exactly what PCA removes.
* `CUST_ID` is an identifier, not a behaviour — it gets dropped.

## 3. Preprocessing

### 3.1 Missing values

In [ ]:
data = df.drop(columns=['CUST_ID']).copy()

# Median imputation: both columns are heavily skewed, so the mean would be pulled
# upward by the outliers. MINIMUM_PAYMENTS missing most likely means "no minimum
# payment was due", but median is the safer, standard choice.
for col in ['MINIMUM_PAYMENTS', 'CREDIT_LIMIT']:
    if data[col].isna().any():
        med = data[col].median()
        n = data[col].isna().sum()
        data[col] = data[col].fillna(med)
        print(f'{col:<20} filled {n:>4} values with median {med:,.2f}')

print('\nRemaining missing values:', data.isna().sum().sum())

### 3.2 Feature engineering

Raw totals depend on how long the customer has been with the bank. Ratios and monthly
averages describe *behaviour* rather than *tenure*, so they separate customers better.

In [ ]:
USE_ENGINEERED = True   # set to False to cluster on the raw 17 features only

def safe_div(a, b):
    # divide, treating division by zero as 0
    return np.where(b == 0, 0, a / np.where(b == 0, 1, b))

if USE_ENGINEERED:
    data['MONTHLY_AVG_PURCHASE']  = safe_div(data['PURCHASES'], data['TENURE'])
    data['MONTHLY_CASH_ADVANCE']  = safe_div(data['CASH_ADVANCE'], data['TENURE'])
    data['LIMIT_USAGE']           = safe_div(data['BALANCE'], data['CREDIT_LIMIT'])
    data['PAYMENT_MINPAY_RATIO']  = safe_div(data['PAYMENTS'], data['MINIMUM_PAYMENTS'])
    data['AVG_PURCHASE_PER_TRX']  = safe_div(data['PURCHASES'], data['PURCHASES_TRX'])
    data['AVG_CASH_ADV_PER_TRX']  = safe_div(data['CASH_ADVANCE'], data['CASH_ADVANCE_TRX'])

    new_cols = ['MONTHLY_AVG_PURCHASE', 'MONTHLY_CASH_ADVANCE', 'LIMIT_USAGE',
                'PAYMENT_MINPAY_RATIO', 'AVG_PURCHASE_PER_TRX', 'AVG_CASH_ADV_PER_TRX']
    print('Added:', new_cols)
    display(data[new_cols].describe().T)

# clean any inf produced by the ratios
data = data.replace([np.inf, -np.inf], np.nan).fillna(0)
print('\nFeature matrix shape:', data.shape)

### 3.3 Outliers and skew correction

In [ ]:
# How extreme are the outliers before treatment?
plt.figure(figsize=(14, 6))
sns.boxplot(data=data[['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'PAYMENTS',
                       'MINIMUM_PAYMENTS', 'CREDIT_LIMIT']], orient='h', palette='Set2')
plt.xscale('log')
plt.title('Monetary features before treatment (log x-axis)')
plt.tight_layout(); plt.show()

In [ ]:
# 1) Winsorise the extreme tail at the 99th percentile so single customers cannot
#    dominate a cluster centroid, then
# 2) log1p-transform the skewed non-negative columns to compress the range.

data_t = data.copy()

CAP_PCT = 0.99
capped = []
for col in data_t.columns:
    hi = data_t[col].quantile(CAP_PCT)
    n_over = (data_t[col] > hi).sum()
    if n_over > 0:
        data_t[col] = data_t[col].clip(upper=hi)
        capped.append((col, round(hi, 2), n_over))

print(f'Winsorised at the {CAP_PCT:.0%} percentile:')
display(pd.DataFrame(capped, columns=['feature', 'cap_value', 'n_clipped']))

# log1p only where it makes sense: non-negative and still skewed
log_cols = [c for c in data_t.columns
            if data_t[c].min() >= 0 and data_t[c].skew() > 0.75]
data_t[log_cols] = np.log1p(data_t[log_cols])
print('\nlog1p applied to:', log_cols)

In [ ]:
# Skew before vs after
cmp = pd.DataFrame({'before': data[data_t.columns].skew(),
                    'after': data_t.skew()}).sort_values('before', ascending=False)

cmp.plot(kind='barh', figsize=(9, 8), color=['#d1495b', '#2a9d8f'])
plt.axvline(0, c='k', lw=1)
plt.title('Skewness before vs after transformation')
plt.tight_layout(); plt.show()
cmp.round(2)

### 3.4 Scaling

K-Means, PCA, DBSCAN and hierarchical clustering are all distance-based. `BALANCE`
runs into the thousands while `PURCHASES_FREQUENCY` lives in [0, 1], so without
standardisation the balance column alone would decide the clusters. `StandardScaler`
puts every feature at mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(data_t)
X = pd.DataFrame(X, columns=data_t.columns, index=data_t.index)

print('Scaled matrix:', X.shape)
print('Mean ~0 :', np.allclose(X.mean(), 0, atol=1e-9))
print('Std  ~1 :', np.allclose(X.std(ddof=0), 1, atol=1e-9))
X.describe().T[['mean', 'std', 'min', 'max']]

## 4. PCA — Dimensionality Reduction

PCA rotates the correlated features into a new set of uncorrelated components ordered
by how much variance they explain. Two uses here:

1. **Compression** — feed a smaller, de-noised matrix into the clustering algorithms.
2. **Visualisation** — plot 8,950 customers described by 20+ variables on a 2-D page.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X)

evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(range(1, len(evr) + 1), evr, color='#3a7ca5')
axes[0].plot(range(1, len(evr) + 1), evr, 'o-', color='#d1495b')
axes[0].set(xlabel='Principal component', ylabel='Explained variance ratio',
            title='Scree plot')

axes[1].plot(range(1, len(cum) + 1), cum, 'o-', color='#2a9d8f')
for thr, c in [(0.80, '#999'), (0.90, '#666'), (0.95, '#333')]:
    axes[1].axhline(thr, ls='--', lw=1, color=c)
    k = int(np.argmax(cum >= thr) + 1)
    axes[1].annotate(f'{thr:.0%} -> {k} PCs', (k, thr), textcoords='offset points',
                     xytext=(6, -14), fontsize=9)
axes[1].set(xlabel='Number of components', ylabel='Cumulative explained variance',
            title='Cumulative explained variance')

plt.tight_layout(); plt.show()

for thr in (0.80, 0.90, 0.95):
    print(f'{thr:.0%} of variance needs {int(np.argmax(cum >= thr) + 1)} components')

In [ ]:
VARIANCE_TARGET = 0.90          # tweak this if you want a tighter/looser compression

pca = PCA(n_components=VARIANCE_TARGET, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
N_PC = X_pca.shape[1]

print(f'{X.shape[1]} features -> {N_PC} components '
      f'({pca.explained_variance_ratio_.sum():.2%} of variance kept)')

pc_names = [f'PC{i+1}' for i in range(N_PC)]
X_pca_df = pd.DataFrame(X_pca, columns=pc_names, index=X.index)
X_pca_df.head()

In [ ]:
# What does each component actually mean? Loadings = correlation of each original
# feature with each component.
loadings = pd.DataFrame(pca.components_.T, columns=pc_names, index=X.columns)

plt.figure(figsize=(12, 10))
sns.heatmap(loadings, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 7}, linewidths=.4, cbar_kws={'shrink': .7})
plt.title('PCA loadings — how each feature contributes to each component')
plt.tight_layout(); plt.show()

# top drivers of the first few components
for pc in pc_names[:4]:
    top = loadings[pc].abs().sort_values(ascending=False).head(5).index
    print(f'\n{pc}  (explains {pca.explained_variance_ratio_[pc_names.index(pc)]:.1%})')
    for f in top:
        print(f'   {f:<34} {loadings.loc[f, pc]:+.3f}')

In [ ]:
# Biplot: customers in PC1-PC2 space with feature vectors on top
fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(X_pca[:, 0], X_pca[:, 1], s=8, alpha=.25, color='#3a7ca5')

scale = 6
for feat in X.columns:
    x_l, y_l = loadings.loc[feat, 'PC1'] * scale, loadings.loc[feat, 'PC2'] * scale
    if np.hypot(x_l, y_l) < 1.2:      # skip clutter from weak contributors
        continue
    ax.arrow(0, 0, x_l, y_l, color='#d1495b', alpha=.8, head_width=.1, lw=1.2)
    ax.text(x_l * 1.1, y_l * 1.1, feat, fontsize=8, color='#7a1f2e')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA biplot')
ax.axhline(0, c='k', lw=.5); ax.axvline(0, c='k', lw=.5)
plt.tight_layout(); plt.show()

## 5. K-Means Clustering

K-Means needs *k* up front, so we scan a range and look at four diagnostics:

| Metric | What it measures | Good value |
|---|---|---|
| Inertia (WCSS) | within-cluster spread | elbow in the curve |
| Silhouette | separation vs cohesion, in [-1, 1] | **higher** |
| Davies-Bouldin | average similarity between a cluster and its closest neighbour | **lower** |
| Calinski-Harabasz | between/within dispersion ratio | **higher** |

In [ ]:
K_RANGE = range(2, 11)
rows = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(X_pca)
    rows.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(X_pca, lab),
        'davies_bouldin': davies_bouldin_score(X_pca, lab),
        'calinski_harabasz': calinski_harabasz_score(X_pca, lab),
    })
    print(f'k={k:<3} done')

scan = pd.DataFrame(rows).set_index('k')
scan

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
specs = [('inertia', 'Elbow method (WCSS)', 'min', '#3a7ca5'),
         ('silhouette', 'Silhouette score (higher is better)', 'max', '#2a9d8f'),
         ('davies_bouldin', 'Davies-Bouldin (lower is better)', 'min', '#d1495b'),
         ('calinski_harabasz', 'Calinski-Harabasz (higher is better)', 'max', '#e9c46a')]

for ax, (col, title, best, colr) in zip(axes.ravel(), specs):
    ax.plot(scan.index, scan[col], 'o-', color=colr, lw=2)
    ax.set(xlabel='k', ylabel=col, title=title)
    ax.set_xticks(list(K_RANGE))
    if col != 'inertia':
        bk = scan[col].idxmax() if best == 'max' else scan[col].idxmin()
        ax.axvline(bk, ls='--', c='k', lw=1)
        ax.annotate(f'best k={bk}', (bk, scan.loc[bk, col]),
                    textcoords='offset points', xytext=(8, 8), fontsize=9)

plt.suptitle('Choosing k', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Best k by silhouette        :', scan['silhouette'].idxmax())
print('Best k by Davies-Bouldin    :', scan['davies_bouldin'].idxmin())
print('Best k by Calinski-Harabasz :', scan['calinski_harabasz'].idxmax())

In [ ]:
# The metrics often disagree - silhouette tends to favour very small k. Pick the value
# that is both well-scored and gives segments the business can actually act on.
K_FINAL = int(scan['silhouette'].idxmax())    # <-- override manually, e.g. K_FINAL = 4
print('Using K_FINAL =', K_FINAL)

kmeans = KMeans(n_clusters=K_FINAL, n_init=25, random_state=RANDOM_STATE)
labels_km = kmeans.fit_predict(X_pca)

print(f'Silhouette        : {silhouette_score(X_pca, labels_km):.4f}')
print(f'Davies-Bouldin    : {davies_bouldin_score(X_pca, labels_km):.4f}')
print(f'Calinski-Harabasz : {calinski_harabasz_score(X_pca, labels_km):.1f}')
print(f'Inertia           : {kmeans.inertia_:,.1f}')
print('\nCluster sizes:')
print(pd.Series(labels_km).value_counts().sort_index()
        .to_frame('customers').assign(pct=lambda d: (d.customers / len(labels_km) * 100).round(1)))

In [ ]:
# Silhouette diagram - one bar per customer, grouped by cluster
sil_vals = silhouette_samples(X_pca, labels_km)
sil_avg = sil_vals.mean()

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
palette = sns.color_palette('deep', K_FINAL)

for i in range(K_FINAL):
    vals = np.sort(sil_vals[labels_km == i])
    y_upper = y_lower + len(vals)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals,
                     facecolor=palette[i], edgecolor=palette[i], alpha=.8)
    ax.text(-0.05, y_lower + len(vals) / 2, str(i), fontweight='bold')
    y_lower = y_upper + 10

ax.axvline(sil_avg, color='red', ls='--', label=f'mean = {sil_avg:.3f}')
ax.set(xlabel='Silhouette coefficient', ylabel='Customers grouped by cluster',
       title=f'Silhouette plot, k={K_FINAL}')
ax.legend(); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Clusters in PCA space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels_km, palette='deep',
                s=14, alpha=.6, ax=axes[0], legend='full')
centers = kmeans.cluster_centers_
axes[0].scatter(centers[:, 0], centers[:, 1], marker='X', s=280, c='black',
                edgecolor='white', lw=1.5, label='centroid', zorder=5)
axes[0].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
            title='K-Means clusters — PC1 vs PC2')
axes[0].legend(title='cluster')

if N_PC >= 3:
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 2], hue=labels_km, palette='deep',
                    s=14, alpha=.6, ax=axes[1], legend=False)
    axes[1].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                ylabel=f'PC3 ({pca.explained_variance_ratio_[2]:.1%})',
                title='K-Means clusters — PC1 vs PC3')
plt.tight_layout(); plt.show()

In [ ]:
# 3-D view of the first three components
if N_PC >= 3:
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    p = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=labels_km,
                   cmap='tab10', s=10, alpha=.6)
    ax.set(xlabel='PC1', ylabel='PC2', zlabel='PC3',
           title=f'K-Means clusters in 3-D PCA space (k={K_FINAL})')
    ax.view_init(elev=20, azim=45)
    fig.colorbar(p, ax=ax, shrink=.55, label='cluster')
    plt.tight_layout(); plt.show()

## 6. Hierarchical (Agglomerative) Clustering

Hierarchical clustering builds a tree of merges instead of asking for *k* up front.
The dendrogram shows the merge distances — cutting where the vertical jumps are
largest gives a natural number of clusters.

In [ ]:
# Full linkage on 8,950 points is heavy to draw, so sample for the dendrogram
SAMPLE = 2000
idx = np.random.choice(len(X_pca), size=min(SAMPLE, len(X_pca)), replace=False)
Z = linkage(X_pca[idx], method='ward')

plt.figure(figsize=(15, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=9, show_contracted=True)
plt.title(f'Ward dendrogram (sample of {len(idx)} customers, last 30 merges)')
plt.xlabel('Cluster size / sample index')
plt.ylabel('Ward distance')
plt.tight_layout(); plt.show()

# largest gaps between successive merge heights = candidate cut points
heights = Z[-15:, 2]
gaps = np.diff(heights)
print('Biggest jumps suggest cutting into',
      [int(len(heights) - i) for i in np.argsort(gaps)[::-1][:3]], 'clusters')

In [ ]:
agg_rows = []
for k in K_RANGE:
    lab = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_pca)
    agg_rows.append({'k': k,
                     'silhouette': silhouette_score(X_pca, lab),
                     'davies_bouldin': davies_bouldin_score(X_pca, lab)})
agg_scan = pd.DataFrame(agg_rows).set_index('k')
display(agg_scan)

K_AGG = int(agg_scan['silhouette'].idxmax())
agglo = AgglomerativeClustering(n_clusters=K_AGG, linkage='ward')
labels_agg = agglo.fit_predict(X_pca)
print(f'Agglomerative with k={K_AGG}: silhouette={silhouette_score(X_pca, labels_agg):.4f}')
print(pd.Series(labels_agg).value_counts().sort_index())

## 7. DBSCAN — density based

DBSCAN finds arbitrarily-shaped dense regions and labels everything else as noise
(`-1`). It needs `eps` (neighbourhood radius) and `min_samples`. The usual heuristic
for `eps` is the elbow of the sorted k-nearest-neighbour distance curve.

In [ ]:
MIN_SAMPLES = 2 * N_PC          # rule of thumb: 2 x dimensionality

nn = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_pca)
dist, _ = nn.kneighbors(X_pca)
kdist = np.sort(dist[:, -1])

plt.figure(figsize=(9, 5))
plt.plot(kdist, color='#3a7ca5', lw=2)
plt.ylabel(f'Distance to {MIN_SAMPLES}th nearest neighbour')
plt.xlabel('Points sorted by distance')
plt.title('k-distance plot — look for the knee to pick eps')
plt.grid(alpha=.4)
plt.tight_layout(); plt.show()

print('Distance percentiles:')
for p in [90, 95, 97, 99]:
    print(f'  {p}th: {np.percentile(kdist, p):.3f}')

In [ ]:
# Small grid search - DBSCAN is sensitive to both parameters
results = []
for eps in np.round(np.linspace(np.percentile(kdist, 85), np.percentile(kdist, 99), 8), 2):
    for ms in [MIN_SAMPLES // 2, MIN_SAMPLES, MIN_SAMPLES * 2]:
        lab = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_pca)
        n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
        n_noise = int((lab == -1).sum())
        sil = (silhouette_score(X_pca[lab != -1], lab[lab != -1])
               if n_clusters > 1 and (lab != -1).sum() > n_clusters else np.nan)
        results.append({'eps': eps, 'min_samples': ms, 'n_clusters': n_clusters,
                        'noise': n_noise, 'noise_pct': round(n_noise / len(lab) * 100, 1),
                        'silhouette_excl_noise': sil})

dbscan_scan = pd.DataFrame(results)
display(dbscan_scan.sort_values('silhouette_excl_noise', ascending=False).head(12))

In [ ]:
# Keep a usable configuration: 2+ clusters and a sensible amount of noise
cand = dbscan_scan[(dbscan_scan.n_clusters >= 2) & (dbscan_scan.noise_pct < 25)]
if len(cand):
    best = cand.sort_values('silhouette_excl_noise', ascending=False).iloc[0]
    EPS, MS = float(best.eps), int(best.min_samples)
else:
    EPS, MS = float(np.percentile(kdist, 95)), MIN_SAMPLES
    print('No configuration hit the criteria - falling back to the 95th percentile eps.')

dbscan = DBSCAN(eps=EPS, min_samples=MS)
labels_db = dbscan.fit_predict(X_pca)
n_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f'eps={EPS}, min_samples={MS} -> {n_db} clusters, '
      f'{(labels_db == -1).sum()} noise points '
      f'({(labels_db == -1).mean():.1%})')

plt.figure(figsize=(10, 7))
noise = labels_db == -1
plt.scatter(X_pca[noise, 0], X_pca[noise, 1], c='lightgrey', s=10, label='noise', alpha=.5)
sns.scatterplot(x=X_pca[~noise, 0], y=X_pca[~noise, 1], hue=labels_db[~noise],
                palette='deep', s=14, alpha=.7)
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title(f'DBSCAN — {n_db} clusters + noise')
plt.legend(title='cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 8. Gaussian Mixture Model

GMM is the soft/probabilistic cousin of K-Means: instead of hard assignments it fits
overlapping Gaussian components and returns a membership probability per customer.
Model order is chosen with BIC/AIC rather than silhouette.

In [ ]:
gmm_rows = []
for k in K_RANGE:
    gm = GaussianMixture(n_components=k, covariance_type='full',
                         n_init=5, random_state=RANDOM_STATE).fit(X_pca)
    lab = gm.predict(X_pca)
    gmm_rows.append({'k': k, 'bic': gm.bic(X_pca), 'aic': gm.aic(X_pca),
                     'silhouette': silhouette_score(X_pca, lab)})
gmm_scan = pd.DataFrame(gmm_rows).set_index('k')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
gmm_scan[['bic', 'aic']].plot(marker='o', ax=axes[0], color=['#3a7ca5', '#d1495b'])
axes[0].axvline(gmm_scan['bic'].idxmin(), ls='--', c='k', lw=1)
axes[0].set(title=f'BIC / AIC (lowest BIC at k={gmm_scan["bic"].idxmin()})', xlabel='k')

gmm_scan['silhouette'].plot(marker='o', ax=axes[1], color='#2a9d8f')
axes[1].set(title='GMM silhouette', xlabel='k')
plt.tight_layout(); plt.show()
display(gmm_scan)

In [ ]:
K_GMM = int(gmm_scan['bic'].idxmin())
gmm = GaussianMixture(n_components=K_GMM, covariance_type='full',
                      n_init=10, random_state=RANDOM_STATE).fit(X_pca)
labels_gmm = gmm.predict(X_pca)
proba = gmm.predict_proba(X_pca)

print(f'GMM with k={K_GMM}: silhouette={silhouette_score(X_pca, labels_gmm):.4f}')
print(f'Mean assignment confidence: {proba.max(axis=1).mean():.3f}')
print(f'Customers assigned with <60% confidence: {(proba.max(axis=1) < 0.6).sum()}')

plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels_gmm, palette='deep',
                s=14, alpha=.6)
plt.title(f'GMM clusters (k={K_GMM})')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='cluster')
plt.tight_layout(); plt.show()

## 9. Model comparison

In [ ]:
def evaluate(name, lab):
    mask = lab != -1                       # exclude DBSCAN noise from the metrics
    n_cl = len(set(lab[mask]))
    if n_cl < 2:
        return {'model': name, 'n_clusters': n_cl, 'silhouette': np.nan,
                'davies_bouldin': np.nan, 'calinski_harabasz': np.nan,
                'noise_pct': round((~mask).mean() * 100, 1)}
    return {'model': name,
            'n_clusters': n_cl,
            'silhouette': round(silhouette_score(X_pca[mask], lab[mask]), 4),
            'davies_bouldin': round(davies_bouldin_score(X_pca[mask], lab[mask]), 4),
            'calinski_harabasz': round(calinski_harabasz_score(X_pca[mask], lab[mask]), 1),
            'noise_pct': round((~mask).mean() * 100, 1)}

comparison = pd.DataFrame([
    evaluate(f'K-Means (k={K_FINAL})', labels_km),
    evaluate(f'Agglomerative (k={K_AGG})', labels_agg),
    evaluate(f'DBSCAN (eps={EPS})', labels_db),
    evaluate(f'GMM (k={K_GMM})', labels_gmm),
]).set_index('model')

display(comparison)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (col, better) in zip(axes, [('silhouette', 'higher'),
                                    ('davies_bouldin', 'lower'),
                                    ('calinski_harabasz', 'higher')]):
    comparison[col].plot(kind='bar', ax=ax, color='#3a7ca5')
    ax.set_title(f'{col} ({better} is better)')
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
# How much do the partitions agree with each other?
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

algos = {'KMeans': labels_km, 'Agglomerative': labels_agg,
         'DBSCAN': labels_db, 'GMM': labels_gmm}

ari = pd.DataFrame(index=algos, columns=algos, dtype=float)
nmi = pd.DataFrame(index=algos, columns=algos, dtype=float)
for a, la in algos.items():
    for b, lb in algos.items():
        ari.loc[a, b] = adjusted_rand_score(la, lb)
        nmi.loc[a, b] = normalized_mutual_info_score(la, lb)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(ari, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Adjusted Rand Index between algorithms')
sns.heatmap(nmi, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Normalised Mutual Information')
plt.tight_layout(); plt.show()

**Choosing the production model.** K-Means is normally the one to ship here: the
segments are balanced, every customer gets an assignment (no noise bucket), the
centroids are easy to explain to a marketing team, and new customers can be scored
instantly with `.predict()`. The cell below sets `FINAL_LABELS` — point it at another
algorithm if the comparison table above favours one.

In [ ]:
FINAL_MODEL_NAME = 'KMeans'
FINAL_LABELS = labels_km
K_USED = K_FINAL
print(f'Profiling {FINAL_MODEL_NAME} with {K_USED} clusters.')

## 10. Cluster profiling — who is in each segment?

Profiling happens on the **original, untransformed** values. Scaled and log-transformed
numbers are meaningless to a business reader; actual cedis/dollars and frequencies are not.

In [ ]:
profile_df = df.copy()
profile_df['CREDIT_LIMIT'] = profile_df['CREDIT_LIMIT'].fillna(profile_df['CREDIT_LIMIT'].median())
profile_df['MINIMUM_PAYMENTS'] = profile_df['MINIMUM_PAYMENTS'].fillna(profile_df['MINIMUM_PAYMENTS'].median())
profile_df['CLUSTER'] = FINAL_LABELS

# add the engineered ratios back for interpretation
if USE_ENGINEERED:
    for c in new_cols:
        profile_df[c] = data[c].values

sizes = (profile_df['CLUSTER'].value_counts().sort_index()
         .to_frame('customers')
         .assign(percent=lambda d: (d.customers / len(profile_df) * 100).round(1)))
display(sizes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(sizes['customers'], labels=[f'Cluster {i}' for i in sizes.index],
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('deep', len(sizes)),
            wedgeprops={'edgecolor': 'white', 'lw': 2})
axes[0].set_title('Share of customers')
sns.barplot(x=sizes.index, y=sizes['customers'], hue=sizes.index,
            legend=False, palette='deep', ax=axes[1])
axes[1].set(xlabel='Cluster', ylabel='Customers', title='Cluster sizes')
for i, v in enumerate(sizes['customers']):
    axes[1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Mean of every feature per cluster
summary = profile_df.drop(columns=['CUST_ID']).groupby('CLUSTER').mean(numeric_only=True)
summary.insert(0, 'N_CUSTOMERS', profile_df['CLUSTER'].value_counts().sort_index())
display(summary.T.round(2))

In [ ]:
# Same table as an index: how far each cluster sits from the overall average,
# in standard deviations. This is what makes the personas obvious.
feat_cols = [c for c in summary.columns if c != 'N_CUSTOMERS']
overall_mean = profile_df[feat_cols].mean()
overall_std = profile_df[feat_cols].std()
z_profile = ((summary[feat_cols] - overall_mean) / overall_std)

plt.figure(figsize=(13, max(6, len(feat_cols) * .38)))
sns.heatmap(z_profile.T, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size': 8}, linewidths=.5, cbar_kws={'label': 'std devs from overall mean'})
plt.title(f'Cluster profile — {FINAL_MODEL_NAME}, k={K_USED}\n(red = above average, blue = below average)')
plt.xlabel('Cluster')
plt.tight_layout(); plt.show()

In [ ]:
# Auto-generated description of each segment from its strongest deviations
print('=' * 78)
print(f'SEGMENT SUMMARY — {FINAL_MODEL_NAME}, k={K_USED}')
print('=' * 78)

for c in sorted(z_profile.index):
    row = z_profile.loc[c].sort_values()
    n = int(summary.loc[c, 'N_CUSTOMERS'])
    print(f'\nCLUSTER {c}  —  {n:,} customers ({n/len(profile_df):.1%})')
    print('-' * 78)
    print('  Stands out HIGH on:')
    for f, v in row.tail(4)[::-1].items():
        print(f'     {f:<34} {v:+.2f} sd   (mean {summary.loc[c, f]:,.2f})')
    print('  Stands out LOW on:')
    for f, v in row.head(4).items():
        print(f'     {f:<34} {v:+.2f} sd   (mean {summary.loc[c, f]:,.2f})')
    print(f'  Headline numbers: balance {summary.loc[c, "BALANCE"]:,.0f} | '
          f'purchases {summary.loc[c, "PURCHASES"]:,.0f} | '
          f'cash advance {summary.loc[c, "CASH_ADVANCE"]:,.0f} | '
          f'credit limit {summary.loc[c, "CREDIT_LIMIT"]:,.0f} | '
          f'full payment {summary.loc[c, "PRC_FULL_PAYMENT"]:.1%}')

In [ ]:
# Distribution of the key behavioural drivers within each cluster
key_feats = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS',
             'PURCHASES_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'PRC_FULL_PAYMENT',
             'PURCHASES_TRX']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, feat in zip(axes.ravel(), key_feats):
    sns.boxplot(data=profile_df, x='CLUSTER', y=feat, hue='CLUSTER',
                legend=False, palette='deep', showfliers=False, ax=ax)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Feature distributions by cluster (outliers hidden)',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Radar chart - the classic way to present segments to a business audience
radar_feats = ['BALANCE', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES',
               'CASH_ADVANCE', 'PURCHASES_FREQUENCY', 'CREDIT_LIMIT',
               'PAYMENTS', 'PRC_FULL_PAYMENT']

# min-max the cluster means onto 0-1 so the axes are comparable
rad = summary[radar_feats]
rad = (rad - rad.min()) / (rad.max() - rad.min() + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'polar': True})
colors = sns.color_palette('deep', len(rad))
for i, (c, row) in enumerate(rad.iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, 'o-', lw=2, label=f'Cluster {c}', color=colors[i])
    ax.fill(angles, vals, alpha=.10, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n') for f in radar_feats], fontsize=8)
ax.set_yticklabels([])
ax.set_title('Cluster fingerprints (min-max scaled cluster means)', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.1))
plt.tight_layout(); plt.show()

In [ ]:
# Where the money is: revenue-style view of each segment
spend = profile_df.groupby('CLUSTER').agg(
    customers=('CUST_ID', 'count'),
    total_purchases=('PURCHASES', 'sum'),
    total_cash_advance=('CASH_ADVANCE', 'sum'),
    avg_balance=('BALANCE', 'mean'),
    avg_credit_limit=('CREDIT_LIMIT', 'mean'),
)
spend['share_of_purchases'] = (spend.total_purchases / spend.total_purchases.sum() * 100).round(1)
spend['share_of_customers'] = (spend.customers / spend.customers.sum() * 100).round(1)
display(spend.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(spend))
ax.bar(x - .2, spend['share_of_customers'], .4, label='% of customers', color='#3a7ca5')
ax.bar(x + .2, spend['share_of_purchases'], .4, label='% of purchase volume', color='#e9c46a')
ax.set_xticks(x); ax.set_xticklabels([f'Cluster {i}' for i in spend.index])
ax.set_ylabel('%')
ax.set_title('Customer share vs purchase-volume share')
ax.legend()
plt.tight_layout(); plt.show()

### How to write up the personas

Read the z-score heatmap column by column and give each cluster a name. On this
dataset the segments that usually fall out are along these lines — match them to
what your run actually produced rather than copying blindly:

* **Revolvers / cash-advance dependent** — high balance, high cash advance, low purchase frequency, low `PRC_FULL_PAYMENT`. Profitable through interest but the credit-risk group. Action: risk monitoring, balance-transfer or consolidation offers.
* **Transactors** — frequent purchases, low balance, high `PRC_FULL_PAYMENT`. Low interest revenue, high interchange revenue, very low risk. Action: cashback and rewards to keep the card top-of-wallet.
* **High-value / premium** — high credit limit, high one-off purchases, large payments. Action: premium tier, travel benefits, retention focus.
* **Low-activity / dormant** — everything below average. Action: reactivation campaigns, or let them churn if the cost to serve outweighs the return.
* **Installment buyers** — high `INSTALLMENTS_PURCHASES` and `PURCHASES_INSTALLMENTS_FREQUENCY`. Action: buy-now-pay-later and 0% instalment plans.

## 11. Export the model and the labelled data

In [ ]:
# Labelled customer file
output = df_raw.copy()
output['CLUSTER'] = FINAL_LABELS
output['CLUSTER_KMEANS'] = labels_km
output['CLUSTER_AGGLOMERATIVE'] = labels_agg
output['CLUSTER_DBSCAN'] = labels_db
output['CLUSTER_GMM'] = labels_gmm
for i in range(N_PC):
    output[f'PC{i+1}'] = X_pca[:, i]

output.to_csv('customer_segments.csv', index=False)
summary.T.round(3).to_csv('cluster_profile_summary.csv')
comparison.to_csv('model_comparison.csv')
print('Wrote customer_segments.csv, cluster_profile_summary.csv, model_comparison.csv')
output.head()

In [ ]:
# Persist the whole preprocessing + model chain so new customers can be scored later
artifact = {
    'scaler': scaler,
    'pca': pca,
    'kmeans': kmeans,
    'gmm': gmm,
    'feature_names': list(data_t.columns),
    'log_cols': log_cols,
    'cap_percentile': CAP_PCT,
    'caps': {c: float(data[c].quantile(CAP_PCT)) for c in data.columns},
    'use_engineered': USE_ENGINEERED,
    'k': K_FINAL,
}
joblib.dump(artifact, 'clustering_model.joblib')
print('Saved clustering_model.joblib')

In [ ]:
def predict_segment(new_df, artifact_path='clustering_model.joblib'):
    # Score unseen customers with the saved pipeline.
    # new_df must have the same raw columns as CC GENERAL.csv (CUST_ID optional).
    art = joblib.load(artifact_path)
    d = new_df.drop(columns=['CUST_ID'], errors='ignore').copy()

    for col in ['MINIMUM_PAYMENTS', 'CREDIT_LIMIT']:
        if col in d:
            d[col] = d[col].fillna(d[col].median())

    if art['use_engineered']:
        sd = lambda a, b: np.where(b == 0, 0, a / np.where(b == 0, 1, b))
        d['MONTHLY_AVG_PURCHASE'] = sd(d['PURCHASES'], d['TENURE'])
        d['MONTHLY_CASH_ADVANCE'] = sd(d['CASH_ADVANCE'], d['TENURE'])
        d['LIMIT_USAGE'] = sd(d['BALANCE'], d['CREDIT_LIMIT'])
        d['PAYMENT_MINPAY_RATIO'] = sd(d['PAYMENTS'], d['MINIMUM_PAYMENTS'])
        d['AVG_PURCHASE_PER_TRX'] = sd(d['PURCHASES'], d['PURCHASES_TRX'])
        d['AVG_CASH_ADV_PER_TRX'] = sd(d['CASH_ADVANCE'], d['CASH_ADVANCE_TRX'])

    d = d.replace([np.inf, -np.inf], np.nan).fillna(0)
    for c, cap in art['caps'].items():
        if c in d:
            d[c] = d[c].clip(upper=cap)
    d[art['log_cols']] = np.log1p(d[art['log_cols']])
    d = d[art['feature_names']]

    z = art['scaler'].transform(d)
    p = art['pca'].transform(z)
    return art['kmeans'].predict(p)

# sanity check on the first 5 training rows - should match the fitted labels
check = predict_segment(df_raw.head(5))
print('predicted:', check)
print('fitted   :', labels_km[:5])

In [ ]:
# Download everything (Colab only)
try:
    from google.colab import files
    for f in ['customer_segments.csv', 'cluster_profile_summary.csv',
              'model_comparison.csv', 'clustering_model.joblib']:
        files.download(f)
except ImportError:
    print('Not running in Colab - files are in the working directory.')

## Conclusion

**What was done**

1. Cleaned 8,950 credit-card records — median-imputed `MINIMUM_PAYMENTS` (313) and `CREDIT_LIMIT` (1).
2. Engineered six behavioural ratios that are independent of tenure.
3. Winsorised at the 99th percentile and log-transformed the skewed monetary columns, then standardised — necessary because every algorithm used here is distance-based.
4. PCA compressed the feature space to the components covering 90% of the variance, removing the redundancy between the strongly correlated purchase and cash-advance columns.
5. Fitted and compared four algorithms — K-Means, Ward agglomerative, DBSCAN, GMM — on internal validation metrics plus cross-algorithm agreement (ARI / NMI).
6. Profiled the winning partition on the original units and exported labels, profiles, and a reusable scoring pipeline.

**Caveats worth stating in the report**

* There is no ground truth. Silhouette and Davies-Bouldin measure geometry, not business usefulness — a slightly worse-scoring *k* that maps to actionable segments is the better choice.
* Silhouette systematically favours small *k*; do not pick *k*=2 purely on that basis if the profiles at *k*=4 or 5 are clearly distinct.
* Winsorising trades information about the very heaviest spenders for stability. If those whales are the point of the exercise, analyse them as a carve-out instead.
* The segmentation reflects one 12-month snapshot; re-fit periodically and check that segment membership is stable before wiring it into campaigns.